# 2일차 실습 1. API 문서 확인 및 테스트

## 실습 목표

이번 실습에서는 FastAPI가 자동으로 제공하는 API 문서와 테스트 흐름을 확인합니다.

단순히 `/docs` 화면을 보는 것에서 끝내지 않고, 노트북 안에서 직접 요청을 보내며 다음 내용을 확인합니다.

- FastAPI 앱과 Endpoint 구성
- 자동 생성되는 API 문서 정보 확인
- Path Parameter 요청 테스트
- Query Parameter 요청 테스트
- Request Body 요청 테스트
- 정상 요청과 검증 실패 요청 비교

## 1. 라이브러리 불러오기

실습에 필요한 라이브러리를 불러옵니다.

In [ ]:
%pip install fastapi uvicorn


In [4]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel
from pprint import pprint
import json

## 2. 실습용 FastAPI 앱 만들기

`/docs`와 요청 테스트에 사용할 FastAPI 앱을 만듭니다.

이번 실습에서는 다음 Endpoint를 사용합니다.

```text
GET  /health
GET  /items/{item_id}
GET  /search
POST /message
```

In [5]:
app = FastAPI()


@app.get("/health")
def health_check():
    return {
        "status": "ok"
    }


@app.get("/items/{item_id}")
def get_item(item_id: int):
    return {
        "item_id": item_id
    }


@app.get("/search")
def search_items(keyword: str = "all", limit: int = 10):
    return {
        "keyword": keyword,
        "limit": limit
    }


class MessageRequest(BaseModel):
    message: str


@app.post("/message")
def create_message(request: MessageRequest):
    return {
        "message": request.message
    }

## 3. TestClient로 요청 보내기

브라우저나 FastAPI 자동 문서 화면을 열기 전에도, `TestClient`를 사용하면 노트북 안에서 API 요청을 직접 보내볼 수 있습니다.

`TestClient`는 FastAPI가 제공하는 테스트 도구로, 실제 서버를 띄우지 않고도 앱을 가상으로 실행해 요청과 응답을 확인합니다.

이렇게 하면 서버 요청과 응답을 코드로 바로 확인할 수 있습니다.

In [6]:
client = TestClient(app)

## 4. 응답을 보기 좋게 출력하는 함수 만들기



In [7]:
def print_api_response(response):
    print("=" * 60)
    print("Status Code:", response.status_code)
    print("-" * 60)

    try:
        print("Response Body:")
        print(json.dumps(response.json(), ensure_ascii=False, indent=2))
    except Exception:
        print("Response Text:")
        print(response.text)

    print("=" * 60)

## 5. `/health` 요청 테스트하기

먼저 입력값이 필요 없는 가장 단순한 API를 테스트합니다.

In [8]:
response = client.get("/health")

print_api_response(response)

Status Code: 200
------------------------------------------------------------
Response Body:
{
  "status": "ok"
}


## 6. Path Parameter 테스트하기 !!!!!!!!!

`/items/{item_id}`는 URL 경로 안의 값을 함수 인자로 받는 Endpoint입니다.

먼저 정상 요청을 보내봅니다.

In [9]:
response = client.get("/items/3")

print_api_response(response)

Status Code: 200
------------------------------------------------------------
Response Body:
{
  "item_id": 3
}


이번에는 숫자가 아닌 값을 넣어 실패 응답을 확인합니다.

In [12]:
response = client.get("/items/1")

print_api_response(response)

Status Code: 200
------------------------------------------------------------
Response Body:
{
  "item_id": 1
}


In [10]:
response = client.get("/items/abc")

print_api_response(response)

Status Code: 422
------------------------------------------------------------
Response Body:
{
  "detail": [
    {
      "type": "int_parsing",
      "loc": [
        "path",
        "item_id"
      ],
      "msg": "Input should be a valid integer, unable to parse string as an integer",
      "input": "abc"
    }
  ]
}


## 7. Query Parameter 테스트하기

`/search`는 Query Parameter를 통해 검색 조건을 받는 Endpoint입니다.

In [11]:
response = client.get("/search", params={
    "keyword": "fastapi",
    "limit": 5
})

print_api_response(response)

Status Code: 200
------------------------------------------------------------
Response Body:
{
  "keyword": "fastapi",
  "limit": 5
}


Query Parameter를 생략하면 기본값이 사용되는지 확인합니다.

In [15]:
response = client.get("/search?keyword=python&limit=3")

print_api_response(response)

Status Code: 200
------------------------------------------------------------
Response Body:
{
  "keyword": "python",
  "limit": 3
}


In [ ]:
response = client.get("/search")

print_api_response(response)

Status Code: 200
------------------------------------------------------------
Response Body:
{
  "keyword": "python",
  "limit": 3
}


## 8. Request Body 테스트하기

`POST /message`는 JSON Body를 입력으로 받는 Endpoint입니다.

먼저 정상 요청을 보내봅니다.

In [16]:
response = client.post("/message", json={
    "message": "hello fastapi"
})

print_api_response(response)

Status Code: 200
------------------------------------------------------------
Response Body:
{
  "message": "hello fastapi"
}


이번에는 필수 필드인 `message`를 빼고 요청해봅니다.

In [17]:
response = client.post("/message", json={})

print_api_response(response)

Status Code: 422
------------------------------------------------------------
Response Body:
{
  "detail": [
    {
      "type": "missing",
      "loc": [
        "body",
        "message"
      ],
      "msg": "Field required",
      "input": {}
    }
  ]
}


## 9. 자동 생성된 API 문서 정보 확인하기

FastAPI는 작성한 Endpoint 정보를 바탕으로 API 문서를 자동 생성합니다.

브라우저에서는 다음 주소로 FastAPI 자동 문서 화면을 확인할 수 있습니다.

```text
http://127.0.0.1:8000/docs
```

노트북에서는 `app.openapi()`를 통해 API 문서의 기반이 되는 OpenAPI 정보를 확인할 수 있습니다.

In [19]:
!python -m uvicorn main:app --reload

INFO:     Will watch for changes in these directories: ['/Users/bkk/프로젝트/yeardream/week3/day2_api실습']
ERROR:    Error loading ASGI app. Could not import module "main".


In [18]:
openapi_schema = app.openapi()

print("API 제목:", openapi_schema["info"]["title"])
print("등록된 Path 목록:")

for path in openapi_schema["paths"].keys():
    print("-", path)

API 제목: FastAPI
등록된 Path 목록:
- /health
- /items/{item_id}
- /search
- /message


각 Path에 어떤 Method가 등록되어 있는지도 확인합니다.

In [20]:
for path, methods in openapi_schema["paths"].items():
    method_names = ", ".join(methods.keys()).upper()
    print(f"{path}: {method_names}")

/health: GET
/items/{item_id}: GET
/search: GET
/message: POST


## 10. 서버 파일로 저장하고 `/docs` 확인하기

아래 셀을 실행하면 지금 만든 FastAPI 앱을 `main.py` 파일로 저장합니다.

In [22]:
!python -m uvicorn main:app --reload

INFO:     Will watch for changes in these directories: ['/Users/bkk/프로젝트/yeardream/week3/day2_api실습']
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     Started reloader process [49174] using StatReload
INFO:     Started server process [49176]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     127.0.0.1:61905 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:61905 - "GET /openapi.json HTTP/1.1" 200 OK
^C
INFO:     Shutting down
INFO:     Finished server process [49176]
INFO:     Stopping reloader process [49174]


In [21]:
%%writefile main.py
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()


@app.get("/health")
def health_check():
    return {
        "status": "ok"
    }


@app.get("/items/{item_id}")
def get_item(item_id: int):
    return {
        "item_id": item_id
    }


@app.get("/search")
def search_items(keyword: str = "all", limit: int = 10):
    return {
        "keyword": keyword,
        "limit": limit
    }


class MessageRequest(BaseModel):
    message: str


@app.post("/message")
def create_message(request: MessageRequest):
    return {
        "message": request.message
    }

Writing main.py


터미널에서 서버를 실행합니다.

```bash
uvicorn main:app --reload
```

브라우저에서 다음 주소로 접속해 FastAPI 자동 문서 화면을 확인합니다.

```text
http://127.0.0.1:8000/docs
```



# TODO 실습

지금까지 TestClient와 `/docs`를 사용해 API 요청과 응답을 확인했습니다.  
아래 TODO는 직접 요청을 보내고 결과를 확인하는 실습입니다.

## TODO 1. `/search` 요청 테스트하기

다음 조건으로 `GET /search` 요청을 보내세요.

```text
keyword = backend
limit = 3
```

기대 응답:

```json
{
  "keyword": "backend",
  "limit": 3
}
```

In [23]:
# TODO 1:
# client.get()을 사용해 /search 요청을 보내고,
# print_api_response()로 결과를 출력하세요.
client = TestClient(app)
response = client.get("/search?keyword=python&limit=3")

print_api_response(response)

Status Code: 200
------------------------------------------------------------
Response Body:
{
  "keyword": "python",
  "limit": 3
}


## TODO 2. `/items/{item_id}` 실패 요청 테스트하기

`GET /items/{item_id}`에 숫자가 아닌 값을 넣어 요청하세요.

예시:

```text
/items/not-number
```



In [24]:
# TODO 2:
# /items/not-number 로 GET 요청을 보내고,
# print_api_response()로 결과를 출력하세요.
response = client.get("/items/not-number")
print_api_response(response)

Status Code: 422
------------------------------------------------------------
Response Body:
{
  "detail": [
    {
      "type": "int_parsing",
      "loc": [
        "path",
        "item_id"
      ],
      "msg": "Input should be a valid integer, unable to parse string as an integer",
      "input": "not-number"
    }
  ]
}


## TODO 3. `/message` 정상 요청과 실패 요청 비교하기

`POST /message`에 대해 정상 요청과 실패 요청을 각각 보내세요.

정상 요청:

```json
{
  "message": "FastAPI docs test"
}
```

실패 요청:

```json
{}
```



In [25]:
# TODO 3-1:
# 정상 요청을 보내고 결과를 출력하세요.
response = client.post("/message", json={
    "message": "FastAPI docs test"
})
print_api_response(response)


print()

# TODO 3-2:
# 실패 요청을 보내고 결과를 출력하세요.

response = client.post("/message", json={
    "message": 111111
})
print_api_response(response)

Status Code: 200
------------------------------------------------------------
Response Body:
{
  "message": "FastAPI docs test"
}

Status Code: 422
------------------------------------------------------------
Response Body:
{
  "detail": [
    {
      "type": "string_type",
      "loc": [
        "body",
        "message"
      ],
      "msg": "Input should be a valid string",
      "input": 111111
    }
  ]
}


## TODO 4. OpenAPI Path 목록 확인하기

`app.openapi()`를 사용해 현재 등록된 Path 목록을 출력하세요.

기대 출력 예시:

```text
/health
/items/{item_id}
/search
/message
```

In [26]:
# TODO 4:
# app.openapi() 결과에서 paths 정보를 꺼내고,
# 등록된 Path 목록을 출력하세요.
openapi = app.openapi()
paths = openapi.get("paths", {})
for path in paths:
    print(path)

/health
/items/{item_id}
/search
/message


In [ ]:
response = client.get("/health")

print_api_response(response)

Status Code: 200
------------------------------------------------------------
Response Body:
{
  "status": "ok"
}


## 실습 정리

이번 실습에서 확인한 내용은 다음과 같습니다.

- TestClient를 사용하면 노트북 안에서 API 요청을 실행할 수 있습니다.
- `/docs`는 FastAPI가 자동으로 제공하는 API 테스트 화면입니다.
- Path Parameter, Query Parameter, Request Body를 각각 테스트할 수 있습니다.
- 잘못된 입력은 Endpoint 처리 전에 검증 실패 응답으로 반환됩니다.
- `app.openapi()`를 통해 자동 문서의 기반 정보를 코드로 확인할 수 있습니다.